In [0]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# =========================
# Funções fornecidas
# =========================
def formatar_brl(valor):
    return f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

def formatar_int(valor):
    return f"{valor:,}".replace(",", ".")

# =========================
# Mapeamento meses PT-BR
# =========================
meses_pt = {
    1: 'jan', 2: 'fev', 3: 'mar', 4: 'abr',
    5: 'mai', 6: 'jun', 7: 'jul', 8: 'ago',
    9: 'set', 10: 'out', 11: 'nov', 12: 'dez'
}

# =========================
# Configuração visual
# =========================
sns.set_theme(style="white")
plt.rcParams["figure.figsize"] = (16, 7)

# =========================
# Leitura da base
# =========================
df = pd.read_csv('data/base_tratada.csv')

# =========================
# Tratamento de data
# =========================
df['purchase_year_month'] = pd.to_datetime(df['purchase_year_month'], format='%Y-%m')

# =========================
# Consolidação por pedido
# =========================
df_pedido = (
    df.groupby('order_id', as_index=False)
      .agg(
          purchase_year_month=('purchase_year_month', 'first'),
          receita_pedido=('payment_value_total', 'first'),
          frete_pedido=('item_freight', 'sum')
      )
)

# =========================
# Agregação mensal
# =========================
df_mensal = (
    df_pedido.groupby('purchase_year_month', as_index=False)
             .agg(
                 quantidade_pedidos=('order_id', 'nunique'),
                 receita_total=('receita_pedido', 'sum'),
                 frete_total=('frete_pedido', 'sum')
             )
             .sort_values('purchase_year_month')
)

# =========================
# Métricas derivadas
# =========================
df_mensal['receita_total_mil'] = df_mensal['receita_total'] / 1000
df_mensal['frete_total_mil'] = df_mensal['frete_total'] / 1000
df_mensal['ticket_medio'] = df_mensal['receita_total'] / df_mensal['quantidade_pedidos']

# Eixo X em português
df_mensal['ano_mes_label'] = df_mensal['purchase_year_month'].apply(
    lambda x: f"{meses_pt[x.month]}/{str(x.year)[2:]}"
)

# =========================
# Cores
# =========================
cor_receita = "#2E86AB"
cor_pedidos = "#F18F01"
cor_frete = "#C73E1D"
cor_ticket = "#6A4C93"

# =========================
# Função de rótulos padrão
# =========================
def adicionar_rotulos(ax, df_plot, coluna, tipo='int', divisor=1):
    for i, valor in enumerate(df_plot[coluna]):
        valor_plot = valor / divisor

        if tipo == 'brl':
            label = formatar_brl(valor_plot)
        else:
            label = formatar_int(int(round(valor_plot)))

        ax.annotate(
            label,
            (i, valor_plot),
            textcoords="offset points",
            xytext=(0, 8),
            ha='center',
            fontsize=10,
            fontweight='semibold',
            alpha=0.95
        )

# =========================
# Função específica Ticket Médio
# =========================
def adicionar_rotulos_ticket(ax, df_plot, coluna):
    for i, valor in enumerate(df_plot[coluna]):
        label = f"R$ {valor:,.1f}".replace(",", "X").replace(".", ",").replace("X", ".")

        ax.annotate(
            label,
            (i, valor),
            textcoords="offset points",
            xytext=(0, 12),
            ha='center',
            va='bottom',
            fontsize=10,
            fontweight='semibold'
        )

# =========================
# Gráfico 1 - Receita
# =========================
plt.figure()
ax = sns.lineplot(
    data=df_mensal,
    x='ano_mes_label',
    y='receita_total_mil',
    color=cor_receita,
    marker='o',
    linewidth=3,
    markersize=8
)

for line in ax.lines:
    line.set_markeredgewidth(2)
    line.set_markeredgecolor('white')

ax.set_title('Evolução Mensal da Receita', fontsize=18, pad=20)
ax.set_xlabel('')
ax.set_ylabel('Receita Total (R$ Mil)')
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: formatar_int(int(round(x)))))
ax.grid(False)
sns.despine()
plt.xticks(rotation=45)

adicionar_rotulos(ax, df_mensal, 'receita_total', divisor=1000)

plt.tight_layout()
plt.show()

# =========================
# Gráfico 2 - Pedidos
# =========================
plt.figure()
ax = sns.lineplot(
    data=df_mensal,
    x='ano_mes_label',
    y='quantidade_pedidos',
    color=cor_pedidos,
    marker='o',
    linewidth=3,
    markersize=8
)

for line in ax.lines:
    line.set_markeredgewidth(2)
    line.set_markeredgecolor('white')

ax.set_title('Evolução Mensal da Quantidade de Pedidos', fontsize=18, pad=20)
ax.set_xlabel('')
ax.set_ylabel('Quantidade de Pedidos')
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: formatar_int(int(round(x)))))
ax.grid(False)
sns.despine()
plt.xticks(rotation=45)

adicionar_rotulos(ax, df_mensal, 'quantidade_pedidos')

plt.tight_layout()
plt.show()

# =========================
# Gráfico 3 - Frete
# =========================
plt.figure()
ax = sns.lineplot(
    data=df_mensal,
    x='ano_mes_label',
    y='frete_total_mil',
    color=cor_frete,
    marker='o',
    linewidth=3,
    markersize=8
)

for line in ax.lines:
    line.set_markeredgewidth(2)
    line.set_markeredgecolor('white')

ax.set_title('Evolução Mensal do Frete', fontsize=18, pad=20)
ax.set_xlabel('')
ax.set_ylabel('Frete Total (R$ Mil)')
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: formatar_int(int(round(x)))))
ax.grid(False)
sns.despine()
plt.xticks(rotation=45)

adicionar_rotulos(ax, df_mensal, 'frete_total', divisor=1000)

plt.tight_layout()
plt.show()

# =========================
# Gráfico 4 - Ticket Médio
# =========================
plt.figure()
ax = sns.lineplot(
    data=df_mensal,
    x='ano_mes_label',
    y='ticket_medio',
    color=cor_ticket,
    marker='o',
    linewidth=3,
    markersize=8
)

for line in ax.lines:
    line.set_markeredgewidth(2)
    line.set_markeredgecolor('white')

ax.set_title('Evolução Mensal do Ticket Médio do Pedido', fontsize=18, pad=20)
ax.set_xlabel('')
ax.set_ylabel('Ticket Médio (R$)')
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: formatar_brl(x)))

ax.grid(False)
sns.despine()
plt.xticks(rotation=45)

# 👇 Rótulos com 1 casa decimal e acima
adicionar_rotulos_ticket(ax, df_mensal, 'ticket_medio')

plt.tight_layout()
plt.show()

# =========================
# Tabela final
# =========================
display(df_mensal)

In [0]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

In [0]:

# =========================
# Funções auxiliares
# =========================
def formatar_brl(valor):
    return f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

def formatar_int(valor):
    return f"{valor:,}".replace(",", ".")

# =========================
# Métricas acumuladas
# =========================
df_mensal['receita_acumulada'] = df_mensal['receita_total'].cumsum()
df_mensal['receita_acumulada_mm'] = df_mensal['receita_acumulada'] / 1_000_000

df_mensal['pedidos_acumulados'] = df_mensal['quantidade_pedidos'].cumsum()
df_mensal['pedidos_acumulados_mil'] = df_mensal['pedidos_acumulados'] / 1000

df_mensal['ticket_medio_acumulado'] = (
    df_mensal['receita_acumulada'] / df_mensal['pedidos_acumulados']
)

# =========================
# Configuração visual
# =========================
sns.set_theme(style="white")
plt.rcParams["figure.figsize"] = (14, 5)

cor_receita = "#2E86AB"
cor_pedidos = "#F18F01"
cor_ticket = "#6A4C93"

# =========================
# MARCOS (25/50/75/100%)
# =========================
total_receita = df_mensal['receita_acumulada'].iloc[-1]

marcos = [0.25, 0.5, 0.75, 1.0]
indices_rotulo = []

for m in marcos:
    idx = (df_mensal['receita_acumulada'] >= total_receita * m).idxmax()
    indices_rotulo.append(idx)

indices_rotulo = sorted(set(indices_rotulo))

# =========================
# GRÁFICO 1 - Receita acumulada
# =========================
plt.figure()
ax = sns.lineplot(
    data=df_mensal,
    x='ano_mes_label',
    y='receita_acumulada_mm',
    color=cor_receita,
    marker='o',
    linewidth=3,
    markersize=7
)

for line in ax.lines:
    line.set_markeredgewidth(1.5)
    line.set_markeredgecolor('white')

ax.set_title('Evolução da Receita Acumulada', fontsize=14, pad=15)
ax.set_ylabel('Receita (R$ MM)', fontweight='bold', color=cor_receita)
ax.set_xlabel('')

plt.xticks(rotation=30, ha='right', fontsize=9)
sns.despine()
ax.grid(False)

for i in indices_rotulo:
    row = df_mensal.loc[i]
    ax.annotate(
        f"{row['receita_acumulada_mm']:.1f}",
        (i, row['receita_acumulada_mm']),
        textcoords="offset points",
        xytext=(0, 10),
        ha='center',
        fontsize=9,
        fontweight='bold',
        color=cor_receita
    )

plt.tight_layout()
plt.show()

# =========================
# GRÁFICO 2 - Pedidos acumulados
# =========================
plt.figure()
ax = sns.lineplot(
    data=df_mensal,
    x='ano_mes_label',
    y='pedidos_acumulados_mil',
    color=cor_pedidos,
    marker='o',
    linewidth=3,
    markersize=7
)

for line in ax.lines:
    line.set_markeredgewidth(1.5)
    line.set_markeredgecolor('white')

ax.set_title('Evolução dos Pedidos Acumulados', fontsize=14, pad=15)
ax.set_ylabel('Pedidos (Mil)', fontweight='bold', color=cor_pedidos)
ax.set_xlabel('')

plt.xticks(rotation=30, ha='right', fontsize=9)
sns.despine()
ax.grid(False)

for i in indices_rotulo:
    row = df_mensal.loc[i]
    ax.annotate(
        f"{row['pedidos_acumulados_mil']:.0f}",
        (i, row['pedidos_acumulados_mil']),
        textcoords="offset points",
        xytext=(0, 10),
        ha='center',
        fontsize=9,
        fontweight='bold',
        color=cor_pedidos
    )

plt.tight_layout()
plt.show()

# =========================
# GRÁFICO 3 - Ticket médio acumulado
# =========================
plt.figure()
ax = sns.lineplot(
    data=df_mensal,
    x='ano_mes_label',
    y='ticket_medio_acumulado',
    color=cor_ticket,
    marker='o',
    linewidth=3,
    markersize=7
)

for line in ax.lines:
    line.set_markeredgewidth(1.5)
    line.set_markeredgecolor('white')

ax.set_title('Evolução do Ticket Médio Acumulado', fontsize=14, pad=15)
ax.set_ylabel('Ticket Médio (R$)', fontweight='bold', color=cor_ticket)
ax.set_xlabel('')

plt.xticks(rotation=30, ha='right', fontsize=9)
sns.despine()
ax.grid(False)

for i in indices_rotulo:
    row = df_mensal.loc[i]
    ax.annotate(
        f"R$ {row['ticket_medio_acumulado']:,.1f}".replace(",", "X").replace(".", ",").replace("X", "."),
        (i, row['ticket_medio_acumulado']),
        textcoords="offset points",
        xytext=(0, 10),
        ha='center',
        fontsize=9,
        fontweight='bold',
        color=cor_ticket
    )

plt.tight_layout()
plt.show()

### **%Participação das 10 top categorias em Receita**

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# ---------------------------
# LEITURA
# ---------------------------
df = pd.read_csv("base_tratada.csv")

# ---------------------------
# TRATAMENTO
# ---------------------------
df["product_category_name"] = df["product_category_name"].fillna("Categoria_desconhecida")

# ---------------------------
# TOP 10 POR RECEITA
# ---------------------------
top10_receita = (
    df.groupby("product_category_name", as_index=False)
      .agg(receita=("item_total", "sum"))
      .sort_values("receita", ascending=False)
      .head(10)
      .copy()
)

# total geral
receita_total = df["item_total"].sum()

# %
top10_receita["pct_total"] = top10_receita["receita"] / receita_total * 100

# R$ MM
top10_receita["receita_mm"] = top10_receita["receita"] / 1_000_000

# ordena para gráfico horizontal (maior em cima)
top10_receita = top10_receita.sort_values("receita_mm", ascending=True)

### **Gráfico horizontal**

In [0]:
plt.figure(figsize=(12, 7))

ax = sns.barplot(
    data=top10_receita,
    y="product_category_name",
    x="receita_mm"
)

plt.title("Top 10 Categorias por Receita Acumulada", fontsize=16, weight="bold")
plt.xlabel("Receita acumulada (R$ MM)")
plt.ylabel("Categoria")

# rótulos
for i, row in top10_receita.reset_index(drop=True).iterrows():
    ax.text(
        row["receita_mm"] + 0.02,
        i,
        f'{row["pct_total"]:.1f}% | R$ {row["receita_mm"]:.2f} MM',
        va="center"
    )

plt.tight_layout()
plt.show()

In [0]:
top10_receita_formatada = top10_receita.copy()

top10_receita_formatada["receita"] = top10_receita_formatada["receita"].map(lambda x: f"R$ {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", "."))
top10_receita_formatada["pct_total"] = top10_receita_formatada["pct_total"].map(lambda x: f"{x:.2f}%")
top10_receita_formatada["receita_mm"] = top10_receita_formatada["receita_mm"].map(lambda x: f"R$ {x:.2f} MM")

print(top10_receita_formatada)

### **1) Código para montar a tabela de concentração**

In [0]:
import pandas as pd

# ---------------------------
# LEITURA
# ---------------------------
df = pd.read_csv("data/base_tratada.csv")

df["product_category_name"] = df["product_category_name"].fillna("Categoria_desconhecida")

# ---------------------------
# RECEITA POR CATEGORIA
# ---------------------------
cat_receita = (
    df.groupby("product_category_name", as_index=False)
      .agg(receita=("item_total", "sum"))
      .sort_values("receita", ascending=False)
      .reset_index(drop=True)
)

# ---------------------------
# PARTICIPAÇÃO E ACUMULADO
# ---------------------------
total_receita = cat_receita["receita"].sum()

cat_receita["pct"] = cat_receita["receita"] / total_receita
cat_receita["pct_acumulado"] = cat_receita["pct"].cumsum()

# ranking
cat_receita["rank"] = cat_receita.index + 1

print(cat_receita.head(20))

### **2) Tabela resumo (Top N x % acumulado)**

In [0]:
# pontos de interesse
pontos = [5, 10, 15, 20, 25, 30]

resumo_concentracao = pd.DataFrame({
    "Top_N": pontos,
    "Receita_Acumulada_%": [
        cat_receita.loc[cat_receita["rank"] == n, "pct_acumulado"].values[0]
        if n <= len(cat_receita)
        else None
        for n in pontos
    ]
})

# formatar %
resumo_concentracao["Receita_Acumulada_%"] = (
    resumo_concentracao["Receita_Acumulada_%"] * 100
).round(2)

print(resumo_concentracao)

### **3) Descobrir automaticamente onde bate 60%, 70%, 80%**

In [0]:
targets = [0.5,0.6, 0.7, 0.8, 0.9]

concentracao_targets = []

for t in targets:
    linha = cat_receita[cat_receita["pct_acumulado"] >= t].iloc[0]
    concentracao_targets.append({
        "Percentual_Receita": f"{int(t*100)}%",
        "Qtd_Categorias": linha["rank"]
    })

df_targets = pd.DataFrame(concentracao_targets)

print(df_targets)

In [0]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.plot(cat_receita["rank"], cat_receita["pct_acumulado"] * 100)

plt.axhline(60, linestyle="--")

plt.title("Curva de Concentração de Receita (Pareto)")
plt.xlabel("Número de Categorias")
plt.ylabel("Receita Acumulada (%)")

plt.grid()
plt.show()

In [0]:
import pandas as pd

df = pd.read_csv("data/base_tratada.csv")

# contar categorias distintas
num_categorias = df["product_category_name"].nunique()

print(f"Número de categorias distintas: {num_categorias}")

### **Mapa de calor com a distribuição dos sellers por região**

In [0]:
pip install folium

In [0]:
%restart_python

In [0]:
import pandas as pd
import folium
from folium.plugins import HeatMap

# ---------------------------
# LEITURA
# ---------------------------
df = pd.read_csv("data/base_tratada.csv")

# ---------------------------
# BASE DE SELLERS (sem duplicar)
# ---------------------------
sellers = (
    df[["seller_id", "seller_lat", "seller_lng"]]
    .dropna()
    .drop_duplicates(subset="seller_id")
)

# ---------------------------
# CRIAR MAPA CENTRALIZADO NO BRASIL
# ---------------------------
mapa = folium.Map(
    location=[-14.2350, -51.9253],  # centro do Brasil
    zoom_start=4
)

# ---------------------------
# HEATMAP
# ---------------------------
heat_data = sellers[["seller_lat", "seller_lng"]].values.tolist()

HeatMap(
    heat_data,
    radius=8,
    blur=6
).add_to(mapa)

# ---------------------------
# SALVAR
# ---------------------------
mapa.save("/Workspace/Users/kaetanokako23@gmail.com/TechChallenge Fase 1/mapa_heatmap_sellers.html")

print("Mapa salvo como mapa_heatmap_sellers.html")

In [0]:
from folium.plugins import MarkerCluster

mapa_cluster = folium.Map(
    location=[-14.2350, -51.9253],
    zoom_start=4
)

marker_cluster = MarkerCluster().add_to(mapa_cluster)

for _, row in sellers.iterrows():
    folium.Marker(
        location=[row["seller_lat"], row["seller_lng"]]
    ).add_to(marker_cluster)

mapa_cluster.save("/Workspace/Users/kaetanokako23@gmail.com/TechChallenge Fase 1/mapa_cluster_sellers.html")

print("Mapa salvo como mapa_cluster_sellers.html")

### **Mapa com peso por volume de vendas do seller**
Peso  = quantidade de pedidos por seller

In [0]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster

# ---------------------------
# LEITURA DA BASE
# ---------------------------
df = pd.read_csv("data/base_tratada.csv")

# ---------------------------
# TRATAMENTO BÁSICO
# ---------------------------
df["seller_lat"] = pd.to_numeric(df["seller_lat"], errors="coerce")
df["seller_lng"] = pd.to_numeric(df["seller_lng"], errors="coerce")
df["item_total"] = pd.to_numeric(df["item_total"], errors="coerce")

df = df.dropna(subset=["seller_id", "seller_lat", "seller_lng", "order_id"]).copy()

# ---------------------------
# AGREGAÇÃO POR SELLER
# ---------------------------
seller_map = (
    df.groupby("seller_id", as_index=False)
      .agg(
          seller_lat=("seller_lat", "mean"),
          seller_lng=("seller_lng", "mean"),
          qtd_pedidos=("order_id", "nunique"),
          receita_total=("item_total", "sum")
      )
)

# ---------------------------
# MAPA BASE
# ---------------------------
mapa_cluster = folium.Map(
    location=[-14.2350, -51.9253],
    zoom_start=4,
    tiles="CartoDB positron"
)

# ---------------------------
# CLUSTER DE SELLERS
# ---------------------------
marker_cluster = MarkerCluster().add_to(mapa_cluster)

for _, row in seller_map.iterrows():
    folium.CircleMarker(
        location=[row["seller_lat"], row["seller_lng"]],
        radius=max(4, row["qtd_pedidos"] ** 0.35),  # peso visual por pedidos
        popup=folium.Popup(
            f"""
            <b>Seller:</b> {row['seller_id']}<br>
            <b>Pedidos:</b> {row['qtd_pedidos']}<br>
            <b>Receita:</b> R$ {row['receita_total']:,.2f}
            """,
            max_width=250
        ),
        tooltip=f"Pedidos: {row['qtd_pedidos']}",
        fill=True,
        fill_opacity=0.7,
        weight=1
    ).add_to(marker_cluster)

# ---------------------------
# SALVAR
# ---------------------------
mapa_cluster.save("/Workspace/Users/kaetanokako23@gmail.com/TechChallenge Fase 1/mapa_cluster_sellers_peso_pedidos.html")

print("Mapa salvo como: mapa_cluster_sellers_peso_pedidos.html")
print(seller_map.head())

### **Calcular % pedidos em atraso**
order_delivered_customer_date > order_estimated_delivery_date

In [0]:
import pandas as pd

# ---------------------------
# LEITURA
# ---------------------------
df = pd.read_csv("data/base_tratada.csv")

# ---------------------------
# DATAS
# ---------------------------
df["order_delivered_customer_date"] = pd.to_datetime(
    df["order_delivered_customer_date"], errors="coerce"
)

df["order_estimated_delivery_date"] = pd.to_datetime(
    df["order_estimated_delivery_date"], errors="coerce"
)

# ---------------------------
# BASE NO NÍVEL DO PEDIDO
# ---------------------------
orders = (
    df.groupby("order_id", as_index=False)
      .agg(
          data_entrega=("order_delivered_customer_date", "max"),
          data_estimada=("order_estimated_delivery_date", "max")
      )
)

# remove pedidos sem data
orders = orders.dropna(subset=["data_entrega", "data_estimada"])

# ---------------------------
# FLAG DE ATRASO
# ---------------------------
orders["atrasado"] = orders["data_entrega"] > orders["data_estimada"]

# ---------------------------
# CÁLCULO DO PERCENTUAL
# ---------------------------
pct_atraso = orders["atrasado"].mean() * 100

print(f"Percentual de pedidos em atraso: {pct_atraso:.2f}%")

In [0]:
# quantidade
qtd_total = len(orders)
qtd_atrasados = orders["atrasado"].sum()

print(f"Total de pedidos: {qtd_total}")
print(f"Pedidos em atraso: {qtd_atrasados}")

# dias de atraso
orders["dias_atraso"] = (
    orders["data_entrega"] - orders["data_estimada"]
).dt.days

media_atraso = orders.loc[orders["atrasado"], "dias_atraso"].mean()

print(f"Atraso médio (dias): {media_atraso:.2f}")

In [0]:
# ---------------------------
# GRÁFICO
# ---------------------------
plt.figure(figsize=(8, 5))

ax = sns.barplot(
    x=count_data.index,
    y=count_data.values
)

plt.title("Pedidos no Prazo vs Atraso", fontsize=14, weight="bold")
plt.xlabel("")
plt.ylabel("Quantidade de pedidos (mil)")

# aumenta espaço no topo
max_value = count_data.values.max()
ax.set_ylim(0, max_value * 1.15)

# eixo Y em mil
ax.set_yticklabels([f"{int(y/1000)}K" for y in ax.get_yticks()])

# rótulos
for i, v in enumerate(count_data.values):
    pct = v / total * 100
    ax.text(
        i,
        v,
        f"{v/1000:.1f}K\n({pct:.1f}%)",
        ha="center",
        va="bottom",
        fontsize=11,
        weight="bold"
    )

plt.tight_layout()
plt.show()

In [0]:
import pandas as pd

# ---------------------------
# LEITURA
# ---------------------------
df = pd.read_csv("data/base_tratada.csv")

# ---------------------------
# DATAS
# ---------------------------
df["order_delivered_customer_date"] = pd.to_datetime(
    df["order_delivered_customer_date"], errors="coerce"
)

df["order_estimated_delivery_date"] = pd.to_datetime(
    df["order_estimated_delivery_date"], errors="coerce"
)

df["order_purchase_timestamp"] = pd.to_datetime(
    df["order_purchase_timestamp"], errors="coerce"
)

# ---------------------------
# CONSOLIDA NO NÍVEL DO PEDIDO
# ---------------------------
orders = (
    df.groupby("order_id", as_index=False)
      .agg(
          data_compra=("order_purchase_timestamp", "min"),
          data_entrega=("order_delivered_customer_date", "max"),
          data_estimada=("order_estimated_delivery_date", "max")
      )
)

# remove nulos
orders = orders.dropna(subset=["data_compra", "data_entrega", "data_estimada"])

# ---------------------------
# FLAG DE ATRASO
# ---------------------------
orders["atrasado"] = orders["data_entrega"] > orders["data_estimada"]

# ---------------------------
# CRIA ANO/MÊS
# ---------------------------
orders["year_month"] = orders["data_compra"].dt.to_period("M").astype(str)
orders["year_month_date"] = pd.to_datetime(orders["year_month"])

In [0]:
monthly = (
    orders.groupby("year_month_date", as_index=False)
      .agg(
          total_pedidos=("order_id", "count"),
          pedidos_atrasados=("atrasado", "sum")
      )
)

monthly = monthly.sort_values("year_month_date")

In [0]:
monthly["total_acum"] = monthly["total_pedidos"].cumsum()
monthly["atrasados_acum"] = monthly["pedidos_atrasados"].cumsum()

monthly["pct_atraso_acum"] = monthly["atrasados_acum"] / monthly["total_acum"] * 100

In [0]:
monthly = monthly[
    (monthly["year_month_date"] >= "2016-10-01") &
    (monthly["year_month_date"] <= "2018-08-01")
]

In [0]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")

plt.figure(figsize=(12, 6))

sns.lineplot(
    data=monthly,
    x="year_month_date",
    y="pct_atraso_acum",
    marker="o",
    linewidth=2
)

plt.title("Evolução do % de Pedidos em Atraso (Acumulado)", fontsize=14, weight="bold")
plt.xlabel("Ano/Mês")
plt.ylabel("% de pedidos em atraso")

plt.gca().yaxis.set_major_formatter(
    plt.FuncFormatter(lambda y, _: f"{y:.1f}%")
)

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## **Análise de Lead time**
Postagem x Entrega (Lead time) - Quanto maior o lead time, pior a avaliação do cliente?

In [0]:
import pandas as pd
import numpy as np

# ---------------------------
# LEITURA
# ---------------------------
df = pd.read_csv("data/base_tratada.csv")

# ---------------------------
# DATAS
# ---------------------------
df["order_delivered_carrier_date"] = pd.to_datetime(df["order_delivered_carrier_date"], errors="coerce")
df["order_delivered_customer_date"] = pd.to_datetime(df["order_delivered_customer_date"], errors="coerce")

# ---------------------------
# LEAD TIME (em dias)
# ---------------------------
df["lead_time_dias"] = (
    df["order_delivered_customer_date"] - df["order_delivered_carrier_date"]
).dt.days

# remove valores inválidos
df = df[(df["lead_time_dias"] >= 0) & (df["lead_time_dias"] <= 60)]

In [0]:
orders = (
    df.groupby("order_id", as_index=False)
      .agg(
          lead_time_dias=("lead_time_dias", "mean"),
          review_score=("review_score_mean", "mean")
      )
)

In [0]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")

plt.figure(figsize=(10, 6))

sns.regplot(
    data=orders,
    x="lead_time_dias",
    y="review_score",
    scatter_kws={"alpha": 0.2},
    line_kws={"color": "red"}
)

plt.title("Relação entre Lead Time e Avaliação do Cliente", fontsize=14, weight="bold")
plt.xlabel("Lead Time (dias)")
plt.ylabel("Review médio")

plt.tight_layout()
plt.show()

In [0]:
corr = orders["lead_time_dias"].corr(orders["review_score"])
print(f"Correlação: {corr:.3f}")

In [0]:
# cria faixas de lead time
bins = [0, 2, 5, 10, 20, 60]
labels = ["0-2", "3-5", "6-10", "11-20", "20+"]

orders["faixa_lead"] = pd.cut(
    orders["lead_time_dias"],
    bins=bins,
    labels=labels
)

lead_analysis = (
    orders.groupby("faixa_lead", as_index=False)
      .agg(
          review_medio=("review_score", "mean"),
          qtd_pedidos=("order_id", "count")
      )
)

print(lead_analysis)

In [0]:
plt.figure(figsize=(10, 5))

sns.barplot(
    data=lead_analysis,
    x="faixa_lead",
    y="review_medio"
)

plt.title("Avaliação Média por Faixa de Lead Time", fontsize=14, weight="bold")
plt.xlabel("Faixa de Lead Time (dias)")
plt.ylabel("Review médio")

plt.ylim(1, 5)

plt.tight_layout()
plt.show()

## **Distribuição das avaliações por quantidade de clientes**

In [0]:
import pandas as pd

# leitura
df = pd.read_csv("data/base_tratada.csv")

# base no nível do pedido
orders = (
    df.groupby("order_id", as_index=False)
      .agg(
          review_score=("review_score_mean", "mean"),
          customer_id=("customer_id", "first")
      )
)

In [0]:
bins = [0, 2, 3, 4, 5]
labels = ["Muito ruim (0-2)", "Ruim (2-3)", "Bom (3-4)", "Excelente (4-5)"]

orders_seller["faixa_review"] = pd.cut(
    orders_seller["review_score"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

In [0]:
# definição das faixas
bins = [0, 2, 3, 4, 5]
labels = ["Muito ruim (0-2)", "Ruim (2-3)", "Bom (3-4)", "Excelente (4-5)"]

orders["faixa_review"] = pd.cut(
    orders["review_score"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

In [0]:
review_dist = (
    orders.groupby("faixa_review", as_index=False)
      .agg(
          qtd_clientes=("customer_id", "nunique")
      )
)

In [0]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")

total = review_dist["qtd_clientes"].sum()

plt.figure(figsize=(10, 6))

ax = sns.barplot(
    data=review_dist,
    x="faixa_review",
    y="qtd_clientes",
    order=labels
)

plt.title("Distribuição de Avaliações dos Clientes", fontsize=14, weight="bold")
plt.xlabel("Faixa de avaliação")
plt.ylabel("Quantidade de clientes (mil)")

# 🔥 aumenta espaço no topo
max_value = review_dist["qtd_clientes"].max()
ax.set_ylim(0, max_value * 1.15)

# eixo em mil
ax.set_yticklabels([f"{int(y/1000)}K" for y in ax.get_yticks()])

# rótulos
for i, row in review_dist.iterrows():
    pct = row["qtd_clientes"] / total * 100
    ax.text(
        i,
        row["qtd_clientes"],
        f"{row['qtd_clientes']/1000:.1f}K\n({pct:.1f}%)",
        ha="center",
        va="bottom",
        fontsize=11,
        weight="bold"
    )

plt.tight_layout()
plt.show()

## **% de clientes que fizeram mais de 1 pedido com mesmo seller**

In [0]:
import pandas as pd

df = pd.read_csv("data/base_tratada.csv")

orders_seller = (
    df.groupby(["order_id", "seller_id"], as_index=False)
      .agg(
          customer_unique_id=("customer_unique_id", "first"),
          review_score=("review_score_mean", "mean")
      )
)

In [0]:
bins = [0, 2, 3, 4, 5]
labels = ["Muito ruim (0-2)", "Ruim (2-3)", "Bom (3-4)", "Excelente (4-5)"]

orders_seller["faixa_review"] = pd.cut(
    orders_seller["review_score"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

In [0]:
# quantidade de compras por cliente + seller
cliente_seller = (
    orders_seller.groupby(["customer_unique_id", "seller_id"], as_index=False)
                 .agg(qtd_compras=("order_id", "nunique"))
)

# flag recompra no mesmo seller
cliente_seller["recomprou_mesmo_seller"] = cliente_seller["qtd_compras"] > 1

# merge de volta
orders_seller = orders_seller.merge(
    cliente_seller,
    on=["customer_unique_id", "seller_id"],
    how="left"
)

In [0]:
recompra_analysis = (
    orders_seller.groupby("faixa_review", as_index=False)
                 .agg(
                     clientes=("customer_unique_id", "nunique"),
                     clientes_recompra=("recomprou_mesmo_seller", "sum")
                 )
)

recompra_analysis["pct_recompra"] = (
    recompra_analysis["clientes_recompra"] / recompra_analysis["clientes"] * 100
)

print(recompra_analysis)

In [0]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

ax = sns.barplot(
    data=recompra_analysis,
    x="faixa_review",
    y="pct_recompra",
    order=labels
)

plt.title("Recompra no Mesmo Seller por Faixa de Avaliação", fontsize=14, weight="bold")
plt.xlabel("Faixa de avaliação")
plt.ylabel("% de clientes que recompraram")

ax.set_ylim(0, recompra_analysis["pct_recompra"].max() * 1.2)

for i, row in recompra_analysis.iterrows():
    ax.text(
        i,
        row["pct_recompra"],
        f"{row['pct_recompra']:.1f}%",
        ha="center",
        va="bottom",
        fontsize=11,
        weight="bold"
    )

plt.tight_layout()
plt.show()

## **% de clientes que fizeram mais de 1 pedido nas plataformas**

In [0]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv("data/base_tratada.csv")

# Verifique se existe customer_unique_id
print(df.columns)

In [0]:
orders = (
    df.groupby("order_id", as_index=False)
      .agg(
          customer_unique_id=("customer_unique_id", "first"),
          review_score=("review_score_mean", "mean")
      )
)

bins = [0, 2, 3, 4, 5]
labels = ["Muito ruim (0-2)", "Ruim (2-3)", "Bom (3-4)", "Excelente (4-5)"]

orders["faixa_review"] = pd.cut(
    orders["review_score"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

# quantidade de pedidos por cliente único
pedidos_cliente = (
    orders.groupby("customer_unique_id", as_index=False)
          .agg(qtd_pedidos=("order_id", "nunique"))
)

pedidos_cliente["recomprou"] = pedidos_cliente["qtd_pedidos"] > 1

orders = orders.merge(
    pedidos_cliente[["customer_unique_id", "recomprou"]],
    on="customer_unique_id",
    how="left"
)

recompra_analysis = (
    orders.groupby("faixa_review", as_index=False)
          .agg(
              clientes=("customer_unique_id", "nunique"),
              clientes_recompra=("recomprou", "sum")
          )
)

recompra_analysis["pct_recompra"] = (
    recompra_analysis["clientes_recompra"] / recompra_analysis["clientes"] * 100
)

print(recompra_analysis)

In [0]:
plt.figure(figsize=(10, 6))

ax = sns.barplot(
    data=recompra_analysis,
    x="faixa_review",
    y="pct_recompra",
    order=labels
)

plt.title("Taxa de Recompra por Faixa de Avaliação", fontsize=14, weight="bold")
plt.xlabel("Faixa de avaliação")
plt.ylabel("% de clientes que recompraram")

ax.set_ylim(0, recompra_analysis["pct_recompra"].max() * 1.20)

for i, row in recompra_analysis.iterrows():
    ax.text(
        i,
        row["pct_recompra"],
        f"{row['pct_recompra']:.1f}%",
        ha="center",
        va="bottom",
        fontsize=11,
        weight="bold"
    )

plt.tight_layout()
plt.show()